# Escenario 2 — Tracking server local con SQLite

**MLflow setup**

| Pieza | Aquí |
|---|---|
| Tracking server | **sí**, local (`127.0.0.1:5001`) |
| Backend store | **SQLite** (`mlflow.db`) |
| Artifact store | sistema de archivos local (`./mlartifacts`) |
| Model Registry | **disponible**, porque hay backend de base de datos |

La diferencia con el escenario 1 es **una línea de configuración** en el cliente,
y a cambio aparece el Model Registry. Eso es lo que hay que retener: el código de
entrenamiento no cambia entre escenarios.

Antes de ejecutar, en una terminal aparte y desde la raíz del repositorio:

```bash
make mlflow
# equivale a:
# uv run mlflow server --backend-store-uri sqlite:///mlflow.db \
#   --default-artifact-root ./mlartifacts --host 127.0.0.1 --port 5001
```

Luego abre la UI en <http://127.0.0.1:5001>.

> **Puerto 5001 en todo el curso.** En macOS, AirPlay Receiver ocupa el puerto
> por defecto de `mlflow server` y responde un HTTP 403 que no explica nada. En
> el material del curso el valor viene de `taxi.config.MLFLOW_PORT`; aquí se
> escribe literal porque estos notebooks son deliberadamente autónomos: enseñan
> topologias, no el caso guía.

## Configuracion del tracking URI

Conecta el cliente MLflow al servidor local en el puerto 5001.


In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5001")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

Confirma que la conexión al servidor se estableció correctamente.

## Listado de Experimentos Disponibles

In [ ]:
mlflow.search_experiments()

## Ejecución del Experimento

In [ ]:
import os

import pandas as pd

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_experiment("iris-server-local")

with mlflow.start_run(run_name="logreg_baseline"):
    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42, "max_iter": 200}
    mlflow.log_params(params)

    mlflow.set_tags(
        {
            "scenario": "2_local_tracking_server",
            # Pon tu usuario: es el tag que permite responder "quien corrio esto".
            "developer": os.environ.get("USER", "estudiante"),
            "module": "mlops_tracking",
            "model_family": "logistic_regression",
            "dataset": "iris",
        }
    )

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)

    mlflow.log_metric("accuracy", float(accuracy_score(y, y_pred)))
    mlflow.log_metric("f1_macro", float(f1_score(y, y_pred, average="macro")))

    preds_path = "predictions_logreg.csv"
    pd.DataFrame({"y_true": y, "y_pred": y_pred}).to_csv(preds_path, index=False)
    mlflow.log_artifact(preds_path)

    model_info = mlflow.sklearn.log_model(
        lr,
        name="model",
        input_example=X[[0]],
        registered_model_name="iris-classifier",
    )

    print({"model_uri": model_info.model_uri, "artifact_uri": mlflow.get_artifact_uri()})

## Resultados de la Ejecución

La ejecución genera:
- Un URI único para los artifacts
- Enlaces directos al servidor MLflow para visualizar:
    - La ejecución específica
    - El experimento completo

Ahora debería aparecer:
- "Default" (creado automáticamente)
- "iris-server-local" (nuestro experimento)

In [ ]:
mlflow.search_experiments()

### Interactuando con el model registry

Crea un cliente para interactuar con el servidor MLflow.

In [ ]:
from mlflow.tracking import MlflowClient


client = MlflowClient("http://127.0.0.1:5001")

Listemos los modelos registrados actualmente (si ya corriste el notebook antes, puede que veas múltiples versiones):

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=mlflow.get_tracking_uri())
client.search_registered_models()

## Model Registry: versiones y aliases

El modelo ya quedó registrado con `mlflow.sklearn.log_model(..., registered_model_name=...)`.
Ahora vamos a:

- listar las versiones disponibles,
- asignar los aliases del curso (`candidate` y `champion`),
- y comprobar que el alias apunta a una sola versión.

**Aliases para enrutar, tags para documentar.** Los *stages*
(`None/Staging/Production/Archived`) están **deprecados desde MLflow 2.9.0**: se
muestran más abajo únicamente como contraejemplo, porque aparecen en casi todos
los tutoriales que vas a encontrar. La decisión está en
[`../../../docs/adr/002-aliases-en-vez-de-stages.md`](../../../docs/adr/002-aliases-en-vez-de-stages.md).

In [ ]:
model_name = "iris-classifier"

# search_model_versions, no get_latest_versions: el segundo esta deprecado porque
# su semantica era "la ultima de cada stage", y los stages ya no existen.
versiones = client.search_model_versions(f"name='{model_name}'")
[(mv.version, mv.run_id, mv.tags) for mv in versiones]

In [ ]:
# La version mas reciente por numero. En un CD real el candidato se identifica
# explicitamente por version o run_id: con dos pipelines concurrentes, "la
# ultima" deja de ser deterministica.
ultima = max(versiones, key=lambda mv: int(mv.version))

client.set_registered_model_alias(model_name, "candidate", ultima.version)
# El tag ANTES del alias: si el proceso muere en medio, el estado resultante es
# "validada pero no promovida", que es el seguro.
client.set_model_version_tag(model_name, ultima.version, "validation_status", "passed")
client.set_registered_model_alias(model_name, "champion", ultima.version)

print("aliases:", client.get_registered_model(model_name).aliases)
print("uri de produccion:", f"models:/{model_name}@champion")

### Contraejemplo: la API de stages (no usar)

La celda siguiente usa `transition_model_version_stage`, el sistema **antiguo**.
Está desactivada a propósito y vive aquí solo para que reconozcas el patrón
cuando lo encuentres:

| API de stages (no usar) | Equivalente vigente |
|---|---|
| `transition_model_version_stage(n, v, stage="Production")` | `set_registered_model_alias(n, "champion", v)` |
| `get_latest_versions(n, stages=["Production"])` | `get_model_version_by_alias(n, "champion")` |
| `models:/<n>/Production` | `models:/<n>@champion` |
| `archive_existing_versions=True` | no hace falta: el alias apunta a una sola versión |

El problema no es solo que esté deprecado: un stage mezclaba *qué versión sirve*
con *en qué estado de validación está*, y dos versiones podían quedar en
`Production` a la vez sin que nadie supiera cuál respondía.

In [ ]:
EJECUTAR_CONTRAEJEMPLO = False  # ponlo en True solo para ver el DeprecationWarning

if EJECUTAR_CONTRAEJEMPLO:
    # CONTRAEJEMPLO. Deprecado desde MLflow 2.9.0. No copiar a codigo real.
    client.transition_model_version_stage(
        name=model_name,
        version=ultima.version,
        stage="Staging",
        archive_existing_versions=False,
    )
    print(client.get_model_version(model_name, ultima.version).current_stage)
else:
    print("Contraejemplo desactivado. La via vigente:")
    print(f"  client.set_registered_model_alias('{model_name}', 'champion', '{ultima.version}')")

## Model Registry vs Tracking: diferencias clave

| | **Tracking** | **Model Registry** |
|---|---|---|
| Pregunta que responde | ¿qué se probó y con qué resultado? | ¿qué artefacto sirve tráfico hoy? |
| Unidad | el `run` (una ejecución) | la **versión** de un modelo con nombre |
| Contenido | params, métricas, tags, artifacts | versiones inmutables, aliases, tags de versión |
| Cuándo se usa | durante la experimentación | al desplegar, promover y hacer rollback |
| Mutabilidad | un run terminado no se reescribe | la versión es inmutable; el **alias** se mueve |

Lo que aporta el registry, y que el tracking no puede dar:

1. **Referencia estable**: `models:/iris-classifier@champion` no cambia cuando
   cambia la versión.
2. **Trazabilidad**: cada versión apunta al run que la produjo, y ese run tiene
   los datos y los params.
3. **Rollback por metadatos**: mover el alias a la versión anterior es una
   escritura de metadatos, no un redeploy.
4. **Estado auditable**: el tag `validation_status` dice si esa versión pasó un
   gate, y cuál.

En una frase: **tracking es el laboratorio, registry es la vitrina** — y el alias
es la etiqueta que dice cuál de los frascos de la vitrina se está usando.

-----------

## Diferencias con el escenario 1

| | Escenario 1 | Escenario 2 |
|---|---|---|
| Servidor | no hay | `mlflow server` en `:5001` |
| Backend | archivos (`mlruns/`) | SQLite (`mlflow.db`) |
| Model Registry | no disponible | disponible |
| Colaboración | una persona | varias, en la misma red |
| Código de entrenamiento | **idéntico** | **idéntico** |

## Los límites de este escenario, dichos explícitamente

SQLite es un archivo. Eso implica tres cosas que en clase no se notan y en
producción sí:

1. **Escrituras concurrentes**: SQLite bloquea la base de datos completa. Varios
   pipelines registrando runs a la vez producen `database is locked`.
2. **Artifacts locales**: viven en el disco de la máquina que corre el servidor.
   Si otra máquina pide el modelo, hay que servir ese disco.
3. **Sin backups ni alta disponibilidad**: si se borra el archivo, se borra el
   historial de experimentos y el registry.

Por eso el escenario 3 cambia SQLite por Postgres y el disco por S3. Lo que **no**
cambia es tu código: sigue siendo `set_tracking_uri` y `log_model`.

**Cuándo usar este escenario:** desarrollo local, cursos, prototipos, equipos de
2 a 5 personas que comparten red. **Cuándo no:** cualquier pipeline automatizado
que corra en paralelo.